In [ ]:
import bioc

def inspect_bioc_xml(file_path, num_samples=15):
    with open(file_path, 'r', encoding='utf-8') as f:
        # Load the BioC collection
        collection = bioc.load(f)
    
    print(f"{'TYPE':<15} | {'TEXT AT OFFSET':<25} | {'OFFSET':<10} | {'LENGTH'}")
    print("-" * 65)

    count = 0
    for doc in collection.documents:
        for passage in doc.passages:
            # Passage-level offset
            p_offset = passage.offset
            p_text = passage.text
            
            for ann in passage.annotations:
                if count >= num_samples:
                    return
                
                # BioC annotations can have multiple locations, usually just one
                loc = ann.locations[0]
                # Calculate the relative position within the passage text
                rel_start = loc.offset - p_offset
                rel_end = rel_start + loc.length
                
                # Extract the actual text from the XML's passage
                actual_text = p_text[rel_start:rel_end]
                label = ann.infons.get('type', 'Unknown').upper()
                
                print(f"{label:<15} | {repr(actual_text):<25} | {loc.offset:<10} | {loc.length}")
                count += 1

# Run the inspection
xml_path = r"mts_data.xml"
inspect_bioc_xml(xml_path)

TYPE            | TEXT AT OFFSET            | OFFSET     | LENGTH
-----------------------------------------------------------------
BPERSON         | 'Doctor'                  | 13         | 6
BORGANIZATION   | 'clinic'                  | 51         | 6
BHISTORY        | 'blood'                   | 110        | 5
IHISTORY        | 'pressure'                | 116        | 8
IHISTORY        | 'medicine'                | 125        | 8
BPERSON         | 'Doctor'                  | 136        | 6
BPERSON         | 'Doctor'                  | 158        | 6
IPERSON         | 'Kumar'                   | 165        | 5
BACTION         | 'followed'                | 171        | 8
IACTION         | 'up'                      | 180        | 2
BHISTORY        | 'hypertension'            | 217        | 12
BHISTORY        | 'osteoarthritis'          | 231        | 14
BHISTORY        | 'osteoporosis'            | 247        | 12
BHISTORY        | 'hypothyroidism'          | 261        | 14
BHISTORY  

In [ ]:
import bioc
import re

def final_safe_save(xml_path, output_path):
    with open(xml_path, 'r', encoding='utf-8') as f:
        collection = bioc.load(f)

    with open(output_path, 'w', encoding='utf-8') as f_out:
        for doc in collection.documents:
            for passage in doc.passages:
                p_text = passage.text
                p_offset = passage.offset
                
                char_map = {}
                for ann in passage.annotations:
                    label = re.sub(r'^[BI]', '', ann.infons.get('type', 'ENTITY')).upper()
                    for loc in ann.locations:
                        start, end = loc.offset - p_offset, loc.offset + loc.length - p_offset
                        for i in range(start, end):
                            # Store just the label and the ID
                            char_map[i] = (label, ann.id)

                tokens = re.finditer(r'\w+|[^\w\s]', p_text)
                
                # --- TRACKING VARIABLES FOR GLUE ---
                last_label_type = None
                last_token_end = -1
                last_id = None
                # ------------------------------------
                
                for t in tokens:
                    t_str = t.group(0)
                    t_start, t_end = t.start(), t.end()
                    assigned_tag = "O"
                    current_id = None
                    current_label_type = None

                    hit = next((char_map[i] for i in range(t_start, t_end) if i in char_map), None)
                    
                    if t_str.lower() in ['doctor', 'patient']:
                        assigned_tag = "B-PERSON"
                        current_id = f"ROLE_{t_start}"
                        current_label_type = "PERSON"
                    elif hit:
                        tag_type, ann_id = hit
                        
                        # --- THE GLUE LOGIC ---
                        # If same ID OR (same Label Type AND right next to each other)
                        is_continuation = (ann_id == last_id) or \
                                         (tag_type == last_label_type and (t_start - last_token_end) <= 1)
                        
                        prefix = "I-" if is_continuation else "B-"
                        # ----------------------
                        
                        assigned_tag = f"{prefix}{tag_type}"
                        current_id = ann_id
                        current_label_type = tag_type
                    
                    f_out.write(f"{t_str}\t{assigned_tag}\n")
                    
                    # Update trackers for the next token
                    last_id = current_id
                    last_label_type = current_label_type
                    last_token_end = t_end

                f_out.write("\n") 

# Run the fixed version
final_safe_save(r"data\mts_data.xml", r"data_iob2\mts_data.iob2")